# GPT-5 Mini and Nano Cookbook

**Provider:** OpenAI  |  **Models:** `gpt-5.4-mini` and `gpt-5.4-nano`

**Docs (mini):** https://developers.openai.com/api/docs/models/gpt-5.4-mini  
**Docs (nano):** https://developers.openai.com/api/docs/models/gpt-5.4-nano

This notebook is a single cookbook that works with both models by changing one variable in the setup cell.

## Choosing Between GPT-5.4 Mini and GPT-5.4 Nano

| | GPT-5.4 Mini | GPT-5.4 Nano |
|---|---|---|
| Best for | Higher quality and deeper reasoning | Fastest and lowest-cost paths |
| Typical use | Assistants, analysis, coding help | Routing, classification, quick summaries |
| Tradeoff | More latency/cost than nano | Lower depth than mini |

> Start with `gpt-5.4-mini` for quality. Move to `gpt-5.4-nano` when you need speed and cost efficiency.

## Contents

- Section 0: Setup
- Section 1: Core API usage
  - 1.1 Responses API
  - 1.2 Chat Completions API
  - 1.3 Multi-turn conversation
  - 1.4 Streaming
- Section 2: Structured outputs and tools
  - 2.1 JSON output
  - 2.2 Tool calling
- Section 3: LangChain integration
- Section 4: Mini vs Nano quick comparison

## Section 0 - Setup

In [ ]:
!pip install -Uqq openai langchain langchain-openai pydantic

In [77]:
import os
import json
import time
from getpass import getpass
from openai import OpenAI

MODEL = 'gpt-5.4-mini'
# MODEL = 'gpt-5.4-nano'

api_key = os.getenv('OPENAI_API_KEY', '').strip()
if not api_key:
    try:
        entered = getpass('Enter OPENAI_API_KEY (input hidden): ').strip()
    except EOFError:
        entered = ''
    if entered:
        os.environ['OPENAI_API_KEY'] = entered
        api_key = entered

client = OpenAI(api_key=api_key) if api_key else None

if client is None:
    print('OPENAI_API_KEY not provided. API cells will be skipped.')
else:
    print(f'Model configured: {MODEL}')

Model configured: gpt-5.4-mini


## Section 1 - Core API Usage

### 1.1 Responses API

Use the Responses API for a concise prompt-response flow.

In [69]:
if client is None:
    print('Skipping: set OPENAI_API_KEY to run this example.')
else:
    try:
        resp = client.responses.create(
            model=MODEL,
            input='Give me a 5-point checklist for shipping a Python CLI tool.'
        )
        print(resp.output_text)
    except Exception as e:
        print(f'Responses API call failed: {type(e).__name__}: {e}')

Here’s a practical 5-point checklist for shipping a Python CLI tool:

1. **Package it properly**
   - Use a standard project layout (`src/` layout is preferred).
   - Add `pyproject.toml` with build metadata and dependencies.
   - Define a console entry point so users can run the tool as a command.

2. **Make the CLI user-friendly**
   - Provide `--help`, `--version`, and clear usage text.
   - Use sensible defaults and helpful error messages.
   - Keep commands and flags consistent and intuitive.

3. **Test the command behavior**
   - Unit test core logic separately from the CLI wrapper.
   - Add integration tests that invoke the command line.
   - Test edge cases, invalid input, and different OS/paths if relevant.

4. **Distribute it cleanly**
   - Build wheels and source distributions.
   - Verify installation works with `pip install .` and from your published package.
   - Publish to PyPI or your intended package index.

5. **Polish the release**
   - Include `README.md`, license, 

### 1.2 Chat Completions API

Use this if your app already uses chat-style messages.

In [70]:
if client is None:
    print('Skipping: set OPENAI_API_KEY to run this example.')
else:
    try:
        chat = client.chat.completions.create(
            model=MODEL,
            messages=[
                {'role': 'system', 'content': 'You are a concise technical assistant.'},
                {'role': 'user', 'content': 'Explain vector databases in under 120 words.'}
            ]
        )
        print(chat.choices[0].message.content)
    except Exception as e:
        print(f'Chat Completions call failed: {type(e).__name__}: {e}')

A vector database stores data as numerical embeddings: arrays that capture meaning or features of text, images, audio, etc. Instead of searching for exact keywords, it finds items that are mathematically similar using distance measures like cosine similarity. This makes it useful for semantic search, recommendation systems, chatbots, and retrieval-augmented generation (RAG). Typical steps: convert content into embeddings with a model, store them in the vector DB, then query with a new embedding to retrieve the closest matches. Vector databases often also support metadata filtering, indexing for speed, and hybrid search with traditional keyword methods.


### 1.3 Multi-turn Conversation

In [71]:
if client is None:
    print('Skipping: set OPENAI_API_KEY to run this example.')
else:
    try:
        conversation = [
            {'role': 'system', 'content': 'You are a senior ML engineer mentor.'},
            {'role': 'user', 'content': 'I have 50k labeled support tickets. How do I build intent classification?'}
        ]

        turn1 = client.chat.completions.create(model=MODEL, messages=conversation)
        assistant_msg = turn1.choices[0].message.content
        print('Assistant (turn 1):\n', assistant_msg)

        conversation.append({'role': 'assistant', 'content': assistant_msg})
        conversation.append({'role': 'user', 'content': 'Now reduce this into a 2-week plan with day-by-day milestones.'})

        turn2 = client.chat.completions.create(model=MODEL, messages=conversation)
        print('\nAssistant (turn 2):\n', turn2.choices[0].message.content)
    except Exception as e:
        print(f'Multi-turn call failed: {type(e).__name__}: {e}')

Assistant (turn 1):
 With 50k labeled support tickets, you have enough data to build a solid intent classifier. A good path is:

## 1) Define the intent taxonomy
- Make sure each ticket has **one clear intent label** or a well-defined multi-label schema.
- Merge overly rare or ambiguous classes if needed.
- Aim for labels that are:
  - mutually exclusive when possible
  - business-actionable
  - stable over time

If labels are messy, model performance will suffer more from taxonomy issues than model choice.

## 2) Split the data properly
Use:
- **train**: 70–80%
- **validation**: 10–15%
- **test**: 10–15%

Important:
- stratify by intent
- if tickets are time-dependent, consider a **time-based split** to avoid leakage
- deduplicate near-identical tickets before splitting

## 3) Start with strong baselines
Before deep learning, build:
- **TF-IDF + Logistic Regression**
- **TF-IDF + Linear SVM**

These are fast and often surprisingly strong for intent classification. They also give you a

### 1.4 Streaming

Stream partial tokens for lower perceived latency in interactive UIs.

In [72]:
if client is None:
    print('Skipping: set OPENAI_API_KEY to run this example.')
else:
    try:
        stream = client.chat.completions.create(
            model=MODEL,
            messages=[{'role': 'user', 'content': 'Write a short intro paragraph about model distillation.'}],
            stream=True
        )

        print('Streaming output:\n')
        for chunk in stream:
            delta = chunk.choices[0].delta
            if delta and delta.content:
                print(delta.content, end='')
        print()
    except Exception as e:
        print(f'Streaming call failed: {type(e).__name__}: {e}')

Streaming output:

Model distillation is a technique for compressing a large, powerful machine learning model into a smaller, faster one while preserving much of its performance. In this process, the smaller “student” model learns to imitate the outputs or behavior of a larger “teacher” model, often capturing useful patterns more efficiently than training from scratch. Distillation is widely used when lower latency, reduced memory use, or easier deployment is more important than having the largest possible model.


## Section 2 - Structured Outputs and Tools

### 2.1 JSON Output

Ask for strict JSON and validate in Python.

In [73]:
if client is None:
    print('Skipping: set OPENAI_API_KEY to run this example.')
else:
    try:
        json_resp = client.chat.completions.create(
            model=MODEL,
            response_format={'type': 'json_object'},
            messages=[
                {'role': 'system', 'content': 'Return valid JSON only.'},
                {'role': 'user', 'content': 'Create a JSON with keys: title, audience, key_points (array of 3 strings) for a workshop on prompt engineering.'}
            ]
        )

        payload = json.loads(json_resp.choices[0].message.content)
        print(json.dumps(payload, indent=2))
    except Exception as e:
        print(f'JSON output call failed: {type(e).__name__}: {e}')

{
  "title": "Prompt Engineering Workshop",
  "audience": "Beginners and practitioners who want to improve how they design and refine prompts for AI systems",
  "key_points": [
    "Understanding the basics of clear, specific prompt writing",
    "Using roles, constraints, and examples to guide model responses",
    "Iterating and evaluating prompts to improve consistency and quality"
  ]
}


### 2.2 Tool Calling

Define a tool schema, let the model choose a tool call, then execute it in Python.

In [74]:
def get_weather(city: str, unit: str = 'celsius'):
    fake = {'delhi': 31, 'mumbai': 29, 'bengaluru': 26}
    temp_c = fake.get(city.lower(), 28)
    if unit == 'fahrenheit':
        return {'city': city, 'temperature': round((temp_c * 9 / 5) + 32, 1), 'unit': 'fahrenheit'}
    return {'city': city, 'temperature': temp_c, 'unit': 'celsius'}

tools = [
    {
        'type': 'function',
        'function': {
            'name': 'get_weather',
            'description': 'Get current weather for a city',
            'parameters': {
                'type': 'object',
                'properties': {
                    'city': {'type': 'string'},
                    'unit': {'type': 'string', 'enum': ['celsius', 'fahrenheit']}
                },
                'required': ['city']
            }
        }
    }
]

if client is None:
    print('Skipping: set OPENAI_API_KEY to run this example.')
else:
    try:
        tool_resp = client.chat.completions.create(
            model=MODEL,
            messages=[{'role': 'user', 'content': 'What is the weather in Delhi in fahrenheit?'}],
            tools=tools,
            tool_choice='auto'
        )

        msg = tool_resp.choices[0].message
        if msg.tool_calls:
            call = msg.tool_calls[0]
            args = json.loads(call.function.arguments)
            result = get_weather(**args)
            print('Tool selected:', call.function.name)
            print('Tool args:', args)
            print('Tool result:', result)
        else:
            print('No tool call. Model reply:')
            print(msg.content)
    except Exception as e:
        print(f'Tool-calling example failed: {type(e).__name__}: {e}')

Tool selected: get_weather
Tool args: {'city': 'Delhi', 'unit': 'fahrenheit'}
Tool result: {'city': 'Delhi', 'temperature': 87.8, 'unit': 'fahrenheit'}


## Section 3 - LangChain Integration

In [75]:
if client is None:
    print('Skipping: set OPENAI_API_KEY to run this example.')
else:
    try:
        from langchain_openai import ChatOpenAI

        llm = ChatOpenAI(
            model=MODEL,
            api_key=os.environ['OPENAI_API_KEY'],
            temperature=0
        )

        result = llm.invoke('Give 3 practical guardrails for enterprise LLM apps.')
        print(result.content)
    except Exception as e:
        print(f'LangChain example failed: {type(e).__name__}: {e}')

LangChain example failed: RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


## Section 4 - Mini vs Nano Quick Comparison

Run the same prompt set on both models and compare latency plus response length.

In [76]:
PROMPTS = [
    'Summarize transformer attention in 4 bullet points.',
    'Draft a concise incident postmortem template for backend outages.',
    'Give Python pseudocode for exponential backoff with jitter.'
]
MODELS = ['gpt-5.4-mini', 'gpt-5.4-nano']

if client is None:
    print('Skipping: set OPENAI_API_KEY to run this example.')
else:
    try:
        rows = []
        for model_name in MODELS:
            for i, prompt in enumerate(PROMPTS, start=1):
                t0 = time.perf_counter()
                r = client.responses.create(model=model_name, input=prompt)
                dt_ms = (time.perf_counter() - t0) * 1000
                text = r.output_text or ''
                rows.append({
                    'model': model_name,
                    'prompt_id': i,
                    'latency_ms': round(dt_ms, 1),
                    'chars': len(text),
                    'preview': text[:120].replace('\n', ' ')
                })

        for row in rows:
            print(row)
    except Exception as e:
        print(f'Benchmark example failed: {type(e).__name__}: {e}')

{'model': 'gpt-5.4-mini', 'prompt_id': 1, 'latency_ms': 1968.1, 'chars': 689, 'preview': '- **Attention lets a model focus on relevant tokens:** for each word/token, the transformer decides which other tokens i'}
{'model': 'gpt-5.4-mini', 'prompt_id': 2, 'latency_ms': 2606.3, 'chars': 1706, 'preview': '# Backend Outage Postmortem Template  ## 1) Incident Summary - **Incident title:** - **Date/time started:** - **Date/tim'}
{'model': 'gpt-5.4-mini', 'prompt_id': 3, 'latency_ms': 1571.7, 'chars': 835, 'preview': '```python import time import random  def retry_with_backoff(operation, max_retries=5, base_delay=0.5, max_delay=30.0):  '}
{'model': 'gpt-5.4-nano', 'prompt_id': 1, 'latency_ms': 1832.6, 'chars': 702, 'preview': '- **Token interactions via attention:** Each token is compared to every other token to determine how much it should “pay'}
{'model': 'gpt-5.4-nano', 'prompt_id': 2, 'latency_ms': 5188.3, 'chars': 3013, 'preview': '## Backend Outage Incident Postmortem Template (Concise)

## Notes

- All examples use a single `MODEL` variable so you can switch mini/nano quickly.
- If a specific endpoint or parameter changes in newer SDK versions, update the package and re-run the setup cell.
- Prefer nano for high-throughput low-cost tasks, and mini for stronger reasoning quality.